# Visualising elfe3D_GPR Results and Post Processing

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pyvista as pv
from scipy.interpolate import griddata

base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\3. May"
postprocess_folder = os.path.join(base_folder, "postprocess")
analytical_folder = os.path.join(base_folder, "semi-analytic_100MHz")

z_value = 0.01
z_tol = 0.05
xlim=(-6.5, 6.5)
ylim=(-6.5, 6.5)
z_tol=0.05
dpi=300
grid_resolution=3000
amplitude_cmap='Blues_r'
phase_cmap='RdBu_r'

component = 'Ex' 

In [2]:
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_ani_def.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

elfe_vtk_all = [elfe_vtk_1]

# Extracting Plot Data
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin(amp_elfe_grid_all[0]))
amp_max = np.ceil(np.nanmax(amp_elfe_grid_all[0]))
amp_clim = [amp_min, amp_max]

# Analytical
xi = np.linspace(xlim[0], xlim[1], grid_resolution)
yi = np.linspace(ylim[0], ylim[1], grid_resolution)
grid_x, grid_y = np.meshgrid(xi, yi)

analytical_file = os.path.join(analytical_folder, "air_z0.01_100MHz_xmax5.csv")
analytical_slices = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

x = analytical_slices[:, 0]
y = analytical_slices[:, 1]

x_unique = np.unique(x)
y_unique = np.unique(y)
nx, ny = len(x_unique), len(y_unique)

if len(x) != nx * ny:
    raise ValueError("Mismatch in analytical grid. Expected meshgrid-style structure.")

if component == 'Ex':
    real_index = 4
    imag_index = 5
elif component == 'Ey':
    real_index = 8
    imag_index = 9
elif component == 'Ez':
    real_index = 12
    imag_index = 13
else:
    raise ValueError("Invalid component. Choose from 'Ex', 'Ey', or 'Ez'.")

real_analytical = analytical_slices[:, real_index].reshape((ny, nx))
imag_analytical = analytical_slices[:, imag_index].reshape((ny, nx))

# ----------------------------------------------------------------------
# Extra Process Based on Inference
# ----------------------------------------------------------------------
real_analytical = -1 * real_analytical
imag_analytical = -1 * imag_analytical

complex_analytical = real_analytical + 1j * imag_analytical

# Interpolation
complex_analytical_grid = griddata((x, y), complex_analytical.flatten(), (grid_x, grid_y), method='linear')
amp_analytical_grid = np.log10(np.clip(np.abs(complex_analytical_grid), 1e-60, None))
phase_analytical_grid = np.angle(complex_analytical_grid)  # returns phase in radians (−π to π)
real_analytical_grid = griddata((x, y), real_analytical.flatten(), (grid_x, grid_y), method='linear')
imag_analytical_grid = griddata((x, y), imag_analytical.flatten(), (grid_x, grid_y), method='linear')

In [ ]:
# assume your data is already defined: amp_elfe_grid_all, amp_analytical_grid, ...
dpi = 300
n_entries = len(elfe_vtk_all)
fig, axs = plt.subplots(
    2, 2,
    figsize=(15,15),
    dpi=dpi,
    gridspec_kw={'width_ratios': [1, 1], 'height_ratios': [1, 1], 'wspace':0.05},
    constrained_layout=True
)

# set titles once
axs[0,0].set_title(r'a) Distribution, Amplitude Error', fontsize=18, fontweight='bold')
axs[1,0].set_title('c) Distribution, Phase Error',      fontsize=18, fontweight='bold')
axs[0,1].set_title(r'b) Histogram, Amplitude Error', fontsize=18, fontweight='bold')
axs[1,1].set_title('d) Histogram, Phase Error',        fontsize=18, fontweight='bold')

# fix limits on the images
for ax in (axs[0,0], axs[1,0]):
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_xlabel("x", fontsize=18)
    ax.set_ylabel("y", fontsize=18)
    ax.tick_params(labelsize=18)

for i in range(n_entries):
    diff_amp   = np.abs(10**amp_elfe_grid_all[i] - 10**amp_analytical_grid) / 10**amp_analytical_grid
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid

    im0 = axs[0,0].imshow(diff_amp,   origin='lower', cmap='inferno_r', vmin=0, vmax=1,
                          extent=(xi.min(), xi.max(), yi.min(), yi.max()))
    im2 = axs[1,0].imshow(diff_phase, origin='lower', cmap='RdBu',
                          extent=(xi.min(), xi.max(), yi.min(), yi.max()))

    # add colorbars (constrained_layout will size them neatly)
    cbar0 = fig.colorbar(im0, ax=axs[0,0], pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    cbar1 = fig.colorbar(im2, ax=axs[1,0], pad=0.02)
    cbar1.ax.tick_params(labelsize=18)    
    
    # histograms
    bins_amp = np.linspace(0, 100, 101)
    clipped = np.clip(diff_amp.flatten()*100, 0, 100)
    stda = np.nanstd(clipped)
    mean_amp = np.nanmean(clipped)
    counts, _, _ = axs[0,1].hist(clipped, bins=bins_amp, alpha=0.7)
    axs[0,1].set_xlabel('Difference (%)', fontsize=18)
    axs[0,1].set_ylabel(r'Frequency $\times 10^6$', fontsize=18)
    axs[0,1].tick_params(labelsize=18)
    ax0 = axs[0,1]
    ax0.text(0.98, 0.80, f'STD: {stda:.2f}\nMean: {mean_amp:.2f}', transform=ax0.transAxes, ha='right', fontsize=18)
    yticks = ax0.get_yticks()
    ax0.set_yticklabels([y/1e6 for y in yticks], fontsize=18)

    axs[1,1].hist(diff_phase.flatten(), bins=100, alpha=0.7)
    axs[1,1].set_xlabel('Difference', fontsize=18)
    axs[1,1].set_ylabel(r'Frequency $\times 10^6$', fontsize=18)
    axs[1,1].tick_params(labelsize=18)
    stdp = np.nanstd(diff_phase.flatten())
    mean_phase = np.nanmean(diff_phase.flatten())
    ax1 = axs[1,1]
    ax1.text(0.98, 0.80, f'STD: {stdp:.2f}\nMean: {mean_phase:.2f}', transform=ax1.transAxes, ha='right', fontsize=18)
    yticks1 = ax1.get_yticks()
    ax1.set_yticklabels([y/1e6 for y in yticks1], fontsize=18)

plt.suptitle(r'Error in elfe3D_GPR, Wholespace Air Model', fontsize=27, fontweight='bold')

# save
out = os.path.join(postprocess_folder, "wa_ani_def_err.png")
os.makedirs(postprocess_folder, exist_ok=True)
fig.savefig(out, dpi=dpi)
plt.close(fig)


C:\Users\Knight\AppData\Local\Temp\ipykernel_22700\4219263760.py:54: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax0.set_yticklabels([y/1e6 for y in yticks], fontsize=24)
C:\Users\Knight\AppData\Local\Temp\ipykernel_22700\4219263760.py:65: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax1.set_yticklabels([y/1e6 for y in yticks1], fontsize=24)


In [4]:
# --- Combined Plotting: Field and Error with Title ---
amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'elfe3D_GPR']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()
n_entries = len(elfe_vtk_all)

fig, axs = plt.subplots(2, 3, figsize=(22, 14),
    gridspec_kw={'width_ratios': [1, 1, 1], 'height_ratios': [1, 1], 'wspace':0.05, 'hspace':0.05},
    constrained_layout=True,
    dpi=dpi)
subplot_labels = [['a', 'b', 'c'], ['d', 'e', 'f']]

# --- Field plots (columns 0,1) ---
for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(f"({subplot_labels[0][i]}) {amp_text_list[i]}", fontsize=25, fontweight='medium')
    axs[0, i].set_xlabel("x", fontsize=24)
    axs[0, i].set_ylabel("y", fontsize=24)
    axs[0, i].tick_params(axis='both', which='major', labelsize=24)
    rect1 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=24)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(f"({subplot_labels[1][i]}) {phase_text_list[i]}", fontsize=25, fontweight='medium')
    axs[1, i].set_xlabel("x", fontsize=24)
    axs[1, i].set_ylabel("y", fontsize=24)
    axs[1, i].tick_params(axis='both', which='major', labelsize=24)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=24)

# Analytical solution (column 1)
im2 = axs[0, 1].imshow(amp_analytical_grid, extent=(-6.5, 6.5, -6.5, 6.5), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, 1].set_title(f"({subplot_labels[0][1]}) $\log_{{10}}(|{component}|)$: Analytical", fontsize=25, fontweight='medium')
axs[0, 1].set_xlabel("x", fontsize=24)
axs[0, 1].set_ylabel("y", fontsize=24)
axs[0, 1].set_xlim(-6.5, 6.5)
axs[0, 1].set_ylim(-6.5, 6.5)
axs[0, 1].tick_params(axis='both', which='major', labelsize=24)
cbar2 = fig.colorbar(im2, ax=axs[0, 1], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=24)

im3 = axs[1, 1].imshow(phase_analytical_grid, extent=(-6.5, 6.5, -6.5, 6.5), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, 1].set_title(f"({subplot_labels[1][1]}) Phase: Analytical", fontsize=25, fontweight='medium')
axs[1, 1].set_xlabel("x", fontsize=24)
axs[1, 1].set_ylabel("y", fontsize=24)
axs[1, 1].set_xlim(-6.5, 6.5)
axs[1, 1].set_ylim(-6.5, 6.5)
axs[1, 1].tick_params(axis='both', which='major', labelsize=24)
cbar3 = fig.colorbar(im3, ax=axs[1, 1], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=24)

# --- Error plots (column 2) ---
for i in range(n_entries):
    diff_amp   = np.abs(10**amp_elfe_grid_all[i] - 10**amp_analytical_grid) / 10**amp_analytical_grid
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid

    # Distribution, Amplitude Error
    im4 = axs[0,2].imshow(diff_amp, origin='lower', cmap='inferno_r', vmin=0, vmax=0.25,
                      extent=(-6.5, 6.5, -6.5, 6.5))
    axs[0,2].set_title(f'({subplot_labels[0][2]}) Amplitude: Error Distribution', fontsize=24, fontweight='medium')
    axs[0,2].set_xlabel("x", fontsize=24)
    axs[0,2].set_ylabel("y", fontsize=24)
    axs[0,2].set_xlim(-6.5, 6.5)
    axs[0,2].set_ylim(-6.5, 6.5)
    axs[0,2].tick_params(axis='both', which='major', labelsize=24)
    cbar4 = fig.colorbar(im4, ax=axs[0,2], fraction=0.046, pad=0.02)
    cbar4.ax.tick_params(labelsize=24)

    # Distribution, Phase Error
    im5 = axs[1,2].imshow(diff_phase, origin='lower', cmap='RdBu',
                        extent=(-6.5, 6.5, -6.5, 6.5))
    axs[1,2].set_title(f'({subplot_labels[1][2]}) Phase: Error Distribution', fontsize=24, fontweight='medium')
    axs[1,2].set_xlabel("x", fontsize=24)
    axs[1,2].set_ylabel("y", fontsize=24)
    axs[1,2].set_xlim(-6.5, 6.5)
    axs[1,2].set_ylim(-6.5, 6.5)
    axs[1,2].tick_params(axis='both', which='major', labelsize=24)
    cbar5 = fig.colorbar(im5, ax=axs[1,2], fraction=0.046, pad=0.02)
    cbar5.ax.tick_params(labelsize=24)

# --- Title ---
title = 'The Working PML with the Exact Reciprocal Decay Function'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
fig.subplots_adjust(top=0.92)  # Adjust this value between 0 and 1
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# --- Save ---
filename_combined = "wa_ani_def.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)
os.makedirs(postprocess_folder, exist_ok=True)
fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


<>:43: SyntaxWarning: invalid escape sequence '\l'
<>:43: SyntaxWarning: invalid escape sequence '\l'
C:\Users\Knight\AppData\Local\Temp\ipykernel_7208\2517585782.py:43: SyntaxWarning: invalid escape sequence '\l'
  axs[0, 1].set_title(f"({subplot_labels[0][1]}) $\log_{{10}}(|{component}|)$: Analytical", fontsize=25, fontweight='medium')
C:\Users\Knight\AppData\Local\Temp\ipykernel_7208\2517585782.py:99: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  fig.subplots_adjust(top=0.92)  # Adjust this value between 0 and 1


## Test for the Theoretical and Numerical Formulations
1. Plot all in one for initial view, with PML visible, and big fonts.
2. Plot with 3 orders clipped.
3. Plot difference plots with analytical.

In [2]:
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_ani_def.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

vtk_file_2 = os.path.join(base_folder, "GPR_model_ani_sf.1.vtk")
elfe_vtk_2 = pv.read(vtk_file_2)

vtk_file_3 = os.path.join(base_folder, "GPR_model_ani_diff_ed.1.vtk")
elfe_vtk_3 = pv.read(vtk_file_3)

vtk_file_4 = os.path.join(base_folder, "GPR_model_ani_diff_cen.1.vtk")
elfe_vtk_4 = pv.read(vtk_file_4)

vtk_file_5 = os.path.join(base_folder, "GPR_model_ani_diff_gm.1.vtk")
elfe_vtk_5 = pv.read(vtk_file_5)

elfe_vtk_all = [elfe_vtk_1, elfe_vtk_2, elfe_vtk_3, elfe_vtk_4, elfe_vtk_5]

In [3]:
# Extracting Plot Data
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

In [4]:
phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin(amp_elfe_grid_all[0]))
amp_max = np.ceil(np.nanmax(amp_elfe_grid_all[0]))
amp_clim = [amp_min, amp_max]

In [2]:
# Analytical
xi = np.linspace(xlim[0], xlim[1], grid_resolution)
yi = np.linspace(ylim[0], ylim[1], grid_resolution)
grid_x, grid_y = np.meshgrid(xi, yi)

analytical_file = os.path.join(analytical_folder, "air_z0.01_100MHz_xmax5.csv")
analytical_slices = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

x = analytical_slices[:, 0]
y = analytical_slices[:, 1]

x_unique = np.unique(x)
y_unique = np.unique(y)
nx, ny = len(x_unique), len(y_unique)

if len(x) != nx * ny:
    raise ValueError("Mismatch in analytical grid. Expected meshgrid-style structure.")

if component == 'Ex':
    real_index = 4
    imag_index = 5
elif component == 'Ey':
    real_index = 8
    imag_index = 9
elif component == 'Ez':
    real_index = 12
    imag_index = 13
else:
    raise ValueError("Invalid component. Choose from 'Ex', 'Ey', or 'Ez'.")

real_analytical = analytical_slices[:, real_index].reshape((ny, nx))
imag_analytical = analytical_slices[:, imag_index].reshape((ny, nx))

# ----------------------------------------------------------------------
# Extra Process Based on Inference
# ----------------------------------------------------------------------
real_analytical = -1 * real_analytical
imag_analytical = -1 * imag_analytical

complex_analytical = real_analytical + 1j * imag_analytical

# Interpolation
complex_analytical_grid = griddata((x, y), complex_analytical.flatten(), (grid_x, grid_y), method='linear')
amp_analytical_grid = np.log10(np.clip(np.abs(complex_analytical_grid), 1e-60, None))
phase_analytical_grid = np.angle(complex_analytical_grid)  # returns phase in radians (−π to π)
real_analytical_grid = griddata((x, y), real_analytical.flatten(), (grid_x, grid_y), method='linear')
imag_analytical_grid = griddata((x, y), imag_analytical.flatten(), (grid_x, grid_y), method='linear')

### Plot 1

In [6]:
# --- Plotting ---
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = ['Anisotropic, Default', 'Anisotropic, Explicit Basis', 'Curl-Curl, All Edges', 'Curl-Curl, Centroid', 'Curl-Curl, Geometric Mean']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Comparison of Various PML Implementation Strategies With the Simple Exact Decay Function, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"1_theory_numerical_comparison.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

### Plot 2

In [2]:
# --- Plotting ---
clip=4
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = ['Anisotropic, Default', 'Anisotropic, Explicit Basis', 'Curl-Curl, All Edges', 'Curl-Curl, Centroid', 'Curl-Curl, Geometric Mean']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Clipped Comparison of Various PML Implementation Strategies With the Simple Exact Decay Function, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"2_theory_numerical_comparison_clipped.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

NameError: name 'elfe_vtk_all' is not defined

### Plot 3

In [ ]:
n_entries = len(elfe_vtk_all)
fig, axs = plt.subplots(4, n_entries, figsize=(6.5 * n_entries, 18), dpi=dpi)

amp_text_list = ['Anisotropic, Default', 'Anisotropic, Explicit Basis', 'Curl-Curl, All Edges', 'Curl-Curl, Centroid', 'Curl-Curl, Geometric Mean']
for i, label in enumerate(amp_text_list):
    axs[0, i].set_title(f'Diff Log Amplitude: {label}', fontsize=18, fontweight='bold')
    axs[1, i].set_title(f'% Diff Log Amp: {label}', fontsize=18, fontweight='bold')
    axs[2, i].set_title(f'Diff Phase: {label}', fontsize=18, fontweight='bold')
    axs[3, i].set_title(f'% Diff Phase: {label}', fontsize=18, fontweight='bold')

for i in range(n_entries):
    diff_amp = amp_elfe_grid_all[i] - amp_analytical_grid
    diff_amp_percent = (diff_amp / amp_analytical_grid) * 100
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid
    diff_phase_percent = (diff_phase / phase_analytical_grid) * 100

    diff_amp_lim = np.nanmax(np.abs(diff_amp))
    diff_phase_lim = np.nanmax(np.abs(diff_phase))
    diff_amp_percent_lim = 100
    diff_phase_percent_lim = 100

    extent = (xi.min(), xi.max(), yi.min(), yi.max())
    cmap = "RdBu"

    im0 = axs[0, i].imshow(diff_amp, extent=extent, origin='lower', cmap=cmap, vmin=-diff_amp_lim, vmax=diff_amp_lim)
    fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    im1 = axs[1, i].imshow(diff_amp_percent, extent=extent, origin='lower', cmap=cmap, vmin=-diff_amp_percent_lim, vmax=diff_amp_percent_lim)
    fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    im2 = axs[2, i].imshow(diff_phase, extent=extent, origin='lower', cmap=cmap, vmin=-diff_phase_lim, vmax=diff_phase_lim)
    fig.colorbar(im2, ax=axs[2, i], fraction=0.046, pad=0.02)
    im3 = axs[3, i].imshow(diff_phase_percent, extent=extent, origin='lower', cmap=cmap, vmin=-diff_phase_percent_lim, vmax=diff_phase_percent_lim)
    fig.colorbar(im3, ax=axs[3, i], fraction=0.046, pad=0.02)

    for row in range(4):
        axs[row, i].set_xlabel("x", fontsize=16)
        axs[row, i].set_ylabel("y", fontsize=16)
        axs[row, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.suptitle('Difference Between elfe3D and Analytical Solution for All Implementation Strategies\n2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')

filename_combined = "3_all_methods_analytic_diff.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)
if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)
fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


## Test for the Various Decay Functions

1. Plot comparing all thicknesses.
2. Their Differences with Analytical.
3. Radial Decay Plots.

In [8]:
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_ani_def.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

vtk_file_3 = os.path.join(base_folder, "GPR_model_ani_pol.1.vtk")
elfe_vtk_3 = pv.read(vtk_file_3)

vtk_file_4 = os.path.join(base_folder, "GPR_model_ani_exp.1.vtk")
elfe_vtk_4 = pv.read(vtk_file_4)

elfe_vtk_all = [elfe_vtk_1, elfe_vtk_3, elfe_vtk_4]

In [9]:
# Extracting Plot Data
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

In [10]:
phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin(amp_elfe_grid_all[0]))
amp_max = np.ceil(np.nanmax(amp_elfe_grid_all[0]))
amp_clim = [amp_min, amp_max]

#### Plot 4

In [17]:
# --- Plotting ---
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries, figsize=(6.5 * n_entries, 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = ['Exact Reciprocal', '4th Order Polynomial', 'Exponential']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)


title = f'Comparison of Various PML Decay Functions, {component} Component of EM Field in Air From an x-Directed Dipole'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"4_decay_functions_comparison.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


In [16]:
print(full_path_combined)

F:\Projects\EMGeoInversion\Tests_Thesis\3. May\postprocess\4_decay_functions_comparison.png


#### Plot 5

In [13]:
# --- Plotting ---
clip = 4
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries, figsize=(6.5 * n_entries, 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = ['Simple Reciprocal, Exact', 'Logarithmic, Exact', '4th Order Poly, Inexact', 'Exponential, Inexact']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)


title = f'Comparison of Various PML Decay Functions, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"5_decay_functions_comparison_clipped.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

#### Plot 6

In [14]:
n_entries = len(elfe_vtk_all)
fig, axs = plt.subplots(4, n_entries, figsize=(6.5 * n_entries, 24), dpi=dpi)

amp_text_list = ['Reciprocal', 'Logarithmic', '4th Order Poly', 'Exponential']

for i, label in enumerate(amp_text_list):
    axs[0, i].set_title(f'Diff Log Amplitude: {label}', fontsize=18, fontweight='bold')
    axs[1, i].set_title(f'% Diff Log Amp: {label}', fontsize=18, fontweight='bold')
    axs[2, i].set_title(f'Diff Phase: {label}', fontsize=18, fontweight='bold')
    axs[3, i].set_title(f'% Diff Phase: {label}', fontsize=18, fontweight='bold')

for i in range(n_entries):
    diff_amp = amp_elfe_grid_all[i] - amp_analytical_grid
    diff_amp_percent = (diff_amp / amp_analytical_grid) * 100
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid
    diff_phase_percent = (diff_phase / phase_analytical_grid) * 100

    diff_amp_lim = np.nanmax(np.abs(diff_amp))
    diff_phase_lim = np.nanmax(np.abs(diff_phase))
    diff_amp_percent_lim = 100
    diff_phase_percent_lim = 100

    extent = (xi.min(), xi.max(), yi.min(), yi.max())
    cmap = "RdBu"

    im0 = axs[0, i].imshow(diff_amp, extent=extent, origin='lower', cmap=cmap, vmin=-diff_amp_lim, vmax=diff_amp_lim)
    fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    im1 = axs[1, i].imshow(diff_amp_percent, extent=extent, origin='lower', cmap=cmap, vmin=-diff_amp_percent_lim, vmax=diff_amp_percent_lim)
    fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    im2 = axs[2, i].imshow(diff_phase, extent=extent, origin='lower', cmap=cmap, vmin=-diff_phase_lim, vmax=diff_phase_lim)
    fig.colorbar(im2, ax=axs[2, i], fraction=0.046, pad=0.02)
    im3 = axs[3, i].imshow(diff_phase_percent, extent=extent, origin='lower', cmap=cmap, vmin=-diff_phase_percent_lim, vmax=diff_phase_percent_lim)
    fig.colorbar(im3, ax=axs[3, i], fraction=0.046, pad=0.02)

    for row in range(4):
        axs[row, i].set_xlabel("x", fontsize=16)
        axs[row, i].set_ylabel("y", fontsize=16)
        axs[row, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.suptitle('Difference Between elfe3D and Analytical Solution for All Decay Functions\n2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')

filename_combined = "6_all_decays_analytic_diff.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)
if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)
fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


#### Plots 7, 8, 9

In [15]:
electric_file = os.path.join(base_folder, "out_ani_def", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_ani_log", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_ani_pol", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_ani_exp", "electric_fields_receiver_line.txt")
e_data_elfe_4 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3, e_data_elfe_4]

num_rec_bs=20
num_rec_ef=20
num_rec_ob=20

#### Analytical - Wholespace

In [16]:
# Analytical Data
analytical_file = os.path.join(analytical_folder, "whole_space_receiver_list_0.01_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)
x_analytical_lines = analytical_lines[:, 0]
y_analytical_lines = analytical_lines[:, 1]
abs_analytical_lines = analytical_lines[:, 2]
real_analytical_lines = -1*analytical_lines[:, 4]
imag_analytical_lines = -1*analytical_lines[:, 5]
phase_analytical_lines = np.angle(real_analytical_lines + 1j * imag_analytical_lines)  # returns phase in radians (−π to π)
num_rec_an = 256

In [17]:
def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    for abs_data, phase_data, rec_xi, rec_yi, label in zip(abs_data_list, phase_data_list, rec_x, rec_y, labels):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data, linestyle='-', marker='o', label=label)
        # Phase plot
        axes[1].plot(rec_distance, phase_data, linestyle='-', marker='o', label=label)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.sqrt(np.array(x_an)**2 + np.array(y_an)**2)
    axes[0].plot(rec_distance_an, abs_an, linestyle='--', color='black', label='Analytical Solution')
    axes[1].plot(rec_distance_an, phase_an, linestyle='--', color='black', label='Analytical Solution')

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)
    axes[1].legend(loc='upper right')

    plt.suptitle(f'{type_rec} Receiver Line Data of {component} Component', fontsize=24, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

labels = [
'Simple Reciprocal, Exact',
'Logarithmic, Exact',
'4th Order Poly, Inexact',
'Exponential, Inexact',
]

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)

# Analytical data
x_an_ef = x_analytical_lines[:num_rec_an]
y_an_ef = y_analytical_lines[:num_rec_an]
abs_Ex_an_ef = abs_analytical_lines[:num_rec_an]
phase_Ex_an_ef = phase_analytical_lines[:num_rec_an]

x_an_bs = x_analytical_lines[num_rec_an:2*num_rec_an]
y_an_bs = y_analytical_lines[num_rec_an:2*num_rec_an]
abs_Ex_an_bs = abs_analytical_lines[num_rec_an:2*num_rec_an]
phase_Ex_an_bs = phase_analytical_lines[num_rec_an:2*num_rec_an]

x_an_ob = x_analytical_lines[2*num_rec_an:]
y_an_ob = y_analytical_lines[2*num_rec_an:]
abs_Ex_an_ob = abs_analytical_lines[2*num_rec_an:]
phase_Ex_an_ob = phase_analytical_lines[2*num_rec_an:]

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 7, 'decay_compare', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 8, 'decay_compare', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 9, 'decay_compare', labels)


## Test for Various PML Thicknesses and Spatial Discretization - Air Wholespace Model
1. Plot comparing all thicknesses.
2. Their Differences with Analytical.
3. Radial Decay Plots - Add Analytical to it.

In [7]:
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_ani_def.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

vtk_file_2 = os.path.join(base_folder, "GPR_model_ani_qua_air_small.1.vtk")
elfe_vtk_2 = pv.read(vtk_file_2)

vtk_file_3 = os.path.join(base_folder, "GPR_model_ani_eight_air.1.vtk")
elfe_vtk_3 = pv.read(vtk_file_3)

elfe_vtk_all = [elfe_vtk_1, elfe_vtk_2, elfe_vtk_3]

KeyboardInterrupt: 

In [ ]:
# Extracting Plot Data
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

In [ ]:
phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin(amp_elfe_grid_all[0]))
amp_max = np.ceil(np.nanmax(amp_elfe_grid_all[0]))
amp_clim = [amp_min, amp_max]

#### Plot 10

In [ ]:
# --- Plotting ---
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'1.5 m, 0.5 $\lambda$', r'0.75 m, 0.25 $\lambda$, Fine', r'0.375 m, 0.125 $\lambda$']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Comparison of Various PML Thicknesses, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"10_thicknesses_comparison.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

#### Plot 11

In [22]:
# --- Plotting ---
clip=4
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries+1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'1.5 m, 0.5 $\lambda$', r'0.75 m, 0.25 $\lambda$, Fine', r'0.375 m, 0.125 $\lambda$']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Clipped Comparison of Various PML Thicknesses, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"11_thicknesses_comparison_clipped.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

#### Plot 12

In [23]:
n_entries = len(elfe_vtk_all)
fig, axs = plt.subplots(4, n_entries, figsize=(6.5 * n_entries, 24), dpi=dpi)

amp_text_list = [r'1.5 m, 0.5 $\lambda$', r'0.75 m, 0.25 $\lambda$, Fine', r'0.375 m, 0.125 $\lambda$']

for i, label in enumerate(amp_text_list):
    axs[0, i].set_title(f'Diff Log Amplitude: {label}', fontsize=18, fontweight='bold')
    axs[1, i].set_title(f'% Diff Log Amp: {label}', fontsize=18, fontweight='bold')
    axs[2, i].set_title(f'Diff Phase: {label}', fontsize=18, fontweight='bold')
    axs[3, i].set_title(f'% Diff Phase: {label}', fontsize=18, fontweight='bold')

for i in range(n_entries):
    diff_amp = amp_elfe_grid_all[i] - amp_analytical_grid
    diff_amp_percent = (diff_amp / amp_analytical_grid) * 100
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid
    diff_phase_percent = (diff_phase / phase_analytical_grid) * 100

    diff_amp_lim = np.nanmax(np.abs(diff_amp))
    diff_phase_lim = np.nanmax(np.abs(diff_phase))
    diff_amp_percent_lim = 100
    diff_phase_percent_lim = 100

    extent = (xi.min(), xi.max(), yi.min(), yi.max())
    cmap = "RdBu"

    im0 = axs[0, i].imshow(diff_amp, extent=extent, origin='lower', cmap=cmap, vmin=-diff_amp_lim, vmax=diff_amp_lim)
    fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    im1 = axs[1, i].imshow(diff_amp_percent, extent=extent, origin='lower', cmap=cmap, vmin=-diff_amp_percent_lim, vmax=diff_amp_percent_lim)
    fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    im2 = axs[2, i].imshow(diff_phase, extent=extent, origin='lower', cmap=cmap, vmin=-diff_phase_lim, vmax=diff_phase_lim)
    fig.colorbar(im2, ax=axs[2, i], fraction=0.046, pad=0.02)
    im3 = axs[3, i].imshow(diff_phase_percent, extent=extent, origin='lower', cmap=cmap, vmin=-diff_phase_percent_lim, vmax=diff_phase_percent_lim)
    fig.colorbar(im3, ax=axs[3, i], fraction=0.046, pad=0.02)

    for row in range(4):
        axs[row, i].set_xlabel("x", fontsize=16)
        axs[row, i].set_ylabel("y", fontsize=16)
        axs[row, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.suptitle('Difference Between elfe3D and Analytical Solution for All PML Thicknesses\n2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')

filename_combined = "12_all_thicknesses_analytic_diff.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)
if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)
fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


#### Plots 13, 14, 15

In [24]:
electric_file = os.path.join(base_folder, "out_ani_def", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_ani_qua_air_small", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_ani_eight_air", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3]

num_rec_bs=20
num_rec_ef=20
num_rec_ob=20

In [25]:
def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    for abs_data, phase_data, rec_xi, rec_yi, label in zip(abs_data_list, phase_data_list, rec_x, rec_y, labels):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data, linestyle='-', marker='o', label=label)
        # Phase plot
        axes[1].plot(rec_distance, phase_data, linestyle='-', marker='o', label=label)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.sqrt(np.array(x_an)**2 + np.array(y_an)**2)
    axes[0].plot(rec_distance_an, abs_an, linestyle='--', color='black', label='Analytical Solution')
    axes[1].plot(rec_distance_an, phase_an, linestyle='--', color='black', label='Analytical Solution')

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)
    axes[1].legend(loc='upper right')

    plt.suptitle(f'{type_rec} Receiver Line Data of {component} Component', fontsize=24, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

labels = [r'1.5 m, 0.5 $\lambda$', r'0.375 m, 0.125 $\lambda$']#, r'0.75 m, Finer', '0.375 m Finer']

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)

# Analytical data
x_an_ef = x_analytical_lines[:num_rec_an]
y_an_ef = y_analytical_lines[:num_rec_an]
abs_Ex_an_ef = abs_analytical_lines[:num_rec_an]
phase_Ex_an_ef = phase_analytical_lines[:num_rec_an]

x_an_bs = x_analytical_lines[num_rec_an:2*num_rec_an]
y_an_bs = y_analytical_lines[num_rec_an:2*num_rec_an]
abs_Ex_an_bs = abs_analytical_lines[num_rec_an:2*num_rec_an]
phase_Ex_an_bs = phase_analytical_lines[num_rec_an:2*num_rec_an]

x_an_ob = x_analytical_lines[2*num_rec_an:]
y_an_ob = y_analytical_lines[2*num_rec_an:]
abs_Ex_an_ob = abs_analytical_lines[2*num_rec_an:]
phase_Ex_an_ob = phase_analytical_lines[2*num_rec_an:]

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 13, 'decay_compare', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 14, 'decay_compare', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 15, 'decay_compare', labels)


## Test for PML's Effectiveness Based on its Proximity to the Source

In [26]:
vtk_file_1 = os.path.join(base_folder, "GPR_model_ani_s_01.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)
elfe_vtk_all = [elfe_vtk_1]

# Extracting Plot Data
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

In [27]:
phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin(amp_elfe_grid_all[0]))
amp_max = np.ceil(np.nanmax(amp_elfe_grid_all[0]))
amp_clim = [amp_min, amp_max]

#### Plot 16

In [28]:
# --- Plotting ---
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$'
phase_text = 'Phase'

for i in range(n_entries):
    amp_text_list[i] = amp_text
    phase_text_list[i] = phase_text

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Source Close to PML Boundary - Absorption Check, {component} Component of an Electric Dipole Field in Air'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"16_source_position.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

## Illustrative Example for Half Space

In [29]:
electric_file = os.path.join(base_folder, "out_ani_half_half_air", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_ani_half_half_med", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2]

num_rec_bs=20
num_rec_ef=20
num_rec_ob=20

#### Plots 17, 18, 19

In [30]:
def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    for abs_data, phase_data, rec_xi, rec_yi, label in zip(abs_data_list, phase_data_list, rec_x, rec_y, labels):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data, linestyle='-', marker='o', label=label)
        # Phase plot
        axes[1].plot(rec_distance, phase_data, linestyle='-', marker='o', label=label)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.sqrt(np.array(x_an)**2 + np.array(y_an)**2)
    axes[0].plot(rec_distance_an, abs_an, linestyle='--', color='black', label='Analytical Solution')
    axes[1].plot(rec_distance_an, phase_an, linestyle='--', color='black', label='Analytical Solution')

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)
    axes[1].legend(loc='upper right')

    plt.suptitle(f'Half-Space Response, {type_rec} Receiver Line Data of {component} Component', fontsize=24, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

labels = [r'1.5m PML, Air-Matched Discretization', r'0.75m PML, Medium-Matched Discretization']

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)

# Analytical data
x_an_ef = x_analytical_lines[:num_rec_an]
y_an_ef = y_analytical_lines[:num_rec_an]
abs_Ex_an_ef = abs_analytical_lines[:num_rec_an]
phase_Ex_an_ef = phase_analytical_lines[:num_rec_an]

x_an_bs = x_analytical_lines[num_rec_an:2*num_rec_an]
y_an_bs = y_analytical_lines[num_rec_an:2*num_rec_an]
abs_Ex_an_bs = abs_analytical_lines[num_rec_an:2*num_rec_an]
phase_Ex_an_bs = phase_analytical_lines[num_rec_an:2*num_rec_an]

x_an_ob = x_analytical_lines[2*num_rec_an:]
y_an_ob = y_analytical_lines[2*num_rec_an:]
abs_Ex_an_ob = abs_analytical_lines[2*num_rec_an:]
phase_Ex_an_ob = phase_analytical_lines[2*num_rec_an:]

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 17, 'decay_compare', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 18, 'decay_compare', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 19, 'decay_compare', labels)


## Illustrative Example for Two Layered Model

In [5]:
electric_file = os.path.join(base_folder, "out_ani_4_9_half", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_ani_4_9_half_fine", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_ani_4_9_half_0002", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3]

num_rec_bs=20
num_rec_ef=20
num_rec_ob=20

In [6]:
# Analytical Data
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_9_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = -real_broadside - 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = -real_45deg - 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = -real_endfire - 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)


#### Plots 20, 21, 22

In [10]:
def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    for abs_data, phase_data, rec_xi, rec_yi, label in zip(abs_data_list, phase_data_list, rec_x, rec_y, labels):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data, linestyle='-', marker='o', label=label)
        # Phase plot
        axes[1].plot(rec_distance, phase_data, linestyle='-', marker='o', label=label)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.sqrt(np.array(x_an)**2 + np.array(y_an)**2)
    axes[0].plot(rec_distance_an, abs_an, linestyle='--', color='black', label='Analytical Solution')
    axes[1].plot(rec_distance_an, phase_an, linestyle='--', color='black', label='Analytical Solution')

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)
    axes[1].legend(loc='upper right')

    plt.suptitle(f'Two Layered Model Response, {type_rec} Receiver Line Data of {component} Component', fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

labels = [r'0.5 m PML - 0.004 m3 Volume', r'0.5 m PML - Finer Volume', r'0.5 m PML - 0.0002 m3 Volume']

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)

# Analytical data
x_an_ef = r
y_an_ef = np.zeros_like(r)
abs_Ex_an_ef = abs_endfire
phase_Ex_an_ef = phase_endfire

x_an_bs = np.zeros_like(r)
y_an_bs = r
abs_Ex_an_bs = abs_broadside
phase_Ex_an_bs = phase_broadside

x_an_ob = r
y_an_ob = r
abs_Ex_an_ob = abs_45deg
phase_Ex_an_ob = phase_45deg

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 20, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 21, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 22, '4_9', labels)


## Test for Mimimum Air Space before PML Can be Used - 0.375m

In [17]:
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_ani_def.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

vtk_file_2 = os.path.join(base_folder, "GPR_model_air_10m_025.1.vtk")
elfe_vtk_2 = pv.read(vtk_file_2)

vtk_file_3 = os.path.join(base_folder, "GPR_model_air_10m_050.1.vtk")
elfe_vtk_3 = pv.read(vtk_file_3)

vtk_file_4 = os.path.join(base_folder, "GPR_model_air_10m_100.1.vtk")
elfe_vtk_4 = pv.read(vtk_file_4)

vtk_file_5 = os.path.join(base_folder, "GPR_model_air_10m_150.1.vtk")
elfe_vtk_5 = pv.read(vtk_file_5)

vtk_file_6 = os.path.join(base_folder, "GPR_model_air_10m_200.1.vtk")
elfe_vtk_6 = pv.read(vtk_file_6)

elfe_vtk_all = [elfe_vtk_1, elfe_vtk_2, elfe_vtk_3, elfe_vtk_4, elfe_vtk_5, elfe_vtk_6]

In [18]:
# Extracting Plot Data
xlim = [-5.375, 5.375]
ylim = [-5.375, 5.375]
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin([np.nanmin(grid) for grid in amp_elfe_grid_all if np.any(~np.isnan(grid))]))
amp_max = np.ceil(np.nanmax([np.nanmax(grid) for grid in amp_elfe_grid_all if np.any(~np.isnan(grid))]))
amp_clim = [amp_min, amp_max]

In [19]:
# Analytical
xi = np.linspace(xlim[0], xlim[1], grid_resolution)
yi = np.linspace(ylim[0], ylim[1], grid_resolution)
grid_x, grid_y = np.meshgrid(xi, yi)

analytical_file = os.path.join(analytical_folder, "air_z0.01_100MHz_xmax5.csv")
analytical_slices = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

x = analytical_slices[:, 0]
y = analytical_slices[:, 1]

x_unique = np.unique(x)
y_unique = np.unique(y)
nx, ny = len(x_unique), len(y_unique)

if len(x) != nx * ny:
    raise ValueError("Mismatch in analytical grid. Expected meshgrid-style structure.")

if component == 'Ex':
    real_index = 4
    imag_index = 5
elif component == 'Ey':
    real_index = 8
    imag_index = 9
elif component == 'Ez':
    real_index = 12
    imag_index = 13
else:
    raise ValueError("Invalid component. Choose from 'Ex', 'Ey', or 'Ez'.")

real_analytical = analytical_slices[:, real_index].reshape((ny, nx))
imag_analytical = analytical_slices[:, imag_index].reshape((ny, nx))

# ----------------------------------------------------------------------
# Extra Process Based on Inference
# ----------------------------------------------------------------------
real_analytical = -1 * real_analytical
imag_analytical = -1 * imag_analytical

complex_analytical = real_analytical + 1j * imag_analytical

# Interpolation
complex_analytical_grid = griddata((x, y), complex_analytical.flatten(), (grid_x, grid_y), method='linear')
amp_analytical_grid = np.log10(np.clip(np.abs(complex_analytical_grid), 1e-60, None))
phase_analytical_grid = np.angle(complex_analytical_grid)  # returns phase in radians (−π to π)
real_analytical_grid = griddata((x, y), real_analytical.flatten(), (grid_x, grid_y), method='linear')
imag_analytical_grid = griddata((x, y), imag_analytical.flatten(), (grid_x, grid_y), method='linear')

#### Plot 23

In [23]:
# --- Plotting ---
extents = (-5.375, 5.375, -5.375, 5.375)  # Set the extents for the images
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'5.0 m, Default', '0.25 m Air', '0.5 m Air', '1.0 m Air', '1.5 m Air', '2.0 m Air']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=extents, origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5,-5), 10, 10, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=extents, origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=2, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=extents, origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=extents, origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Comparison of Various Air Space Thicknesses Above and Below the Source - with PML 1/8th of a Wavelength thick, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"23_air_space_comparison.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

#### Plot 24

In [24]:
# --- Plotting ---
clip=4
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'5.0 m, Default', '0.25 m Air', '0.5 m Air', '1.0 m Air', '1.5 m Air', '2.0 m Air']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=extents, origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5,-5),10,10, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=extents, origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=2, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=extents, origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=extents, origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Clipped Comparison of Various Air Space Thicknesses Above and Below the Source - with PML 1/8th of a Wavelength thick, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"24_air_space_comparison_clipped.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

In [26]:
n_entries = len(elfe_vtk_all)
fig, axs = plt.subplots(4, n_entries, figsize=(6.5 * n_entries, 24), dpi=dpi)

amp_text_list = [r'5.0 m, Default', '0.25 m Air', '0.5 m Air', '1.0 m Air', '1.5 m Air', '2.0 m Air']
for i, label in enumerate(amp_text_list):
    axs[0, i].set_title(f'Log of Ratio of Amplitudes: {label}', fontsize=18, fontweight='bold')
    axs[1, i].set_title(f'Normalised Diff, Amplitudes: {label}', fontsize=18, fontweight='bold')
    axs[2, i].set_title(f'Diff Phase: {label}', fontsize=18, fontweight='bold')
    axs[3, i].set_title(f'% Diff Phase: {label}', fontsize=18, fontweight='bold')

for i in range(n_entries):
    diff_amp = amp_elfe_grid_all[i] - amp_analytical_grid
    diff_amp_percent = ((np.power(10, amp_elfe_grid_all[i]) - np.power(10, amp_analytical_grid)) / np.power(10, amp_analytical_grid)) * 100
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid
    diff_phase_percent = (diff_phase / phase_analytical_grid) * 100

    diff_amp_lim = np.nanmax(np.abs(diff_amp))
    diff_phase_lim = np.nanmax(np.abs(diff_phase))
    diff_amp_percent_lim = 100
    diff_phase_percent_lim = 100

    extent = (xi.min(), xi.max(), yi.min(), yi.max())
    cmap = "RdBu"

    im0 = axs[0, i].imshow(diff_amp, extent=extents, origin='lower', cmap=cmap, vmin=-diff_amp_lim, vmax=diff_amp_lim)
    fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    im1 = axs[1, i].imshow(diff_amp_percent, extent=extents, origin='lower', cmap=cmap, vmin=-diff_amp_percent_lim, vmax=diff_amp_percent_lim)
    fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    im2 = axs[2, i].imshow(diff_phase, extent=extents, origin='lower', cmap=cmap, vmin=-diff_phase_lim, vmax=diff_phase_lim)
    fig.colorbar(im2, ax=axs[2, i], fraction=0.046, pad=0.02)
    im3 = axs[3, i].imshow(diff_phase_percent, extent=extents, origin='lower', cmap=cmap, vmin=-diff_phase_percent_lim, vmax=diff_phase_percent_lim)
    fig.colorbar(im3, ax=axs[3, i], fraction=0.046, pad=0.02)

    for row in range(4):
        axs[row, i].set_xlabel("x", fontsize=16)
        axs[row, i].set_ylabel("y", fontsize=16)
        axs[row, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.suptitle('Difference Between elfe3D and Analytical Solution for Various Air Spaces Above and Below the Source - with PML 1/8th of a Wavelength thick\n2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')

filename_combined = "25_air_space_comparison_diff.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)
if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)
fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


## Test for Mimimum Air Space before PML Can be Used - 0.75m

In [28]:
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_ani_def.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

vtk_file_2 = os.path.join(base_folder, "GPR_model_air_10m_025_75.1.vtk")
elfe_vtk_2 = pv.read(vtk_file_2)

vtk_file_3 = os.path.join(base_folder, "GPR_model_air_10m_050_75.1.vtk")
elfe_vtk_3 = pv.read(vtk_file_3)

vtk_file_4 = os.path.join(base_folder, "GPR_model_air_10m_100_75.1.vtk")
elfe_vtk_4 = pv.read(vtk_file_4)

vtk_file_5 = os.path.join(base_folder, "GPR_model_air_10m_150_75.1.vtk")
elfe_vtk_5 = pv.read(vtk_file_5)

vtk_file_6 = os.path.join(base_folder, "GPR_model_air_10m_200_75.1.vtk")
elfe_vtk_6 = pv.read(vtk_file_6)

elfe_vtk_all = [elfe_vtk_1, elfe_vtk_2, elfe_vtk_3, elfe_vtk_4, elfe_vtk_5, elfe_vtk_6]

In [29]:
# Extracting Plot Data
xlim = [-5.75, 5.75]
ylim = [-5.75, 5.75]
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin([np.nanmin(grid) for grid in amp_elfe_grid_all if np.any(~np.isnan(grid))]))
amp_max = np.ceil(np.nanmax([np.nanmax(grid) for grid in amp_elfe_grid_all if np.any(~np.isnan(grid))]))
amp_clim = [amp_min, amp_max]

In [30]:
# Analytical
xi = np.linspace(xlim[0], xlim[1], grid_resolution)
yi = np.linspace(ylim[0], ylim[1], grid_resolution)
grid_x, grid_y = np.meshgrid(xi, yi)

analytical_file = os.path.join(analytical_folder, "air_z0.01_100MHz_xmax5.csv")
analytical_slices = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

x = analytical_slices[:, 0]
y = analytical_slices[:, 1]

x_unique = np.unique(x)
y_unique = np.unique(y)
nx, ny = len(x_unique), len(y_unique)

if len(x) != nx * ny:
    raise ValueError("Mismatch in analytical grid. Expected meshgrid-style structure.")

if component == 'Ex':
    real_index = 4
    imag_index = 5
elif component == 'Ey':
    real_index = 8
    imag_index = 9
elif component == 'Ez':
    real_index = 12
    imag_index = 13
else:
    raise ValueError("Invalid component. Choose from 'Ex', 'Ey', or 'Ez'.")

real_analytical = analytical_slices[:, real_index].reshape((ny, nx))
imag_analytical = analytical_slices[:, imag_index].reshape((ny, nx))

# ----------------------------------------------------------------------
# Extra Process Based on Inference
# ----------------------------------------------------------------------
real_analytical = -1 * real_analytical
imag_analytical = -1 * imag_analytical

complex_analytical = real_analytical + 1j * imag_analytical

# Interpolation
complex_analytical_grid = griddata((x, y), complex_analytical.flatten(), (grid_x, grid_y), method='linear')
amp_analytical_grid = np.log10(np.clip(np.abs(complex_analytical_grid), 1e-60, None))
phase_analytical_grid = np.angle(complex_analytical_grid)  # returns phase in radians (−π to π)
real_analytical_grid = griddata((x, y), real_analytical.flatten(), (grid_x, grid_y), method='linear')
imag_analytical_grid = griddata((x, y), imag_analytical.flatten(), (grid_x, grid_y), method='linear')

#### Plot 26

In [35]:
# --- Plotting ---
extents = (-5.75, 5.75, -5.75, 5.75)  # Set the extents for the images
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'5.0 m, Default', '0.25 m Air', '0.5 m Air', '1.0 m Air', '1.5 m Air', '2.0 m Air']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=extents, origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5,-5), 10, 10, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=extents, origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=2, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=extents, origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=extents, origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Comparison of Various Air Space Thicknesses Above and Below the Source - with PML 1/4th of a Wavelength thick, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"26_air_space_comparison_75.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

#### Plot 27

In [36]:
# --- Plotting ---
clip=4
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'5.0 m, Default', '0.25 m Air', '0.5 m Air', '1.0 m Air', '1.5 m Air', '2.0 m Air']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=extents, origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5,-5),10,10, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=extents, origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=2, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=extents, origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=extents, origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Clipped Comparison of Various Air Space Thicknesses Above and Below the Source - with PML 1/4th of a Wavelength thick, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"27_air_space_comparison_clipped_75.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

#### Plot 28

In [40]:
n_entries = len(elfe_vtk_all)
fig, axs = plt.subplots(4, n_entries, figsize=(6.5 * n_entries, 24), dpi=dpi)

amp_text_list = [r'5.0 m, Default', '0.25 m Air', '0.5 m Air', '1.0 m Air', '1.5 m Air', '2.0 m Air']
for i, label in enumerate(amp_text_list):
    axs[0, i].set_title(f'Log of Ratio of Amplitudes: {label}', fontsize=18, fontweight='bold')
    axs[1, i].set_title(f'Normalised Diff, Amplitudes: {label}', fontsize=18, fontweight='bold')
    axs[2, i].set_title(f'Diff Phase: {label}', fontsize=18, fontweight='bold')
    axs[3, i].set_title(f'% Diff Phase: {label}', fontsize=18, fontweight='bold')

for i in range(n_entries):
    diff_amp = amp_elfe_grid_all[i] - amp_analytical_grid
    diff_amp_percent = ((np.power(10, amp_elfe_grid_all[i]) - np.power(10, amp_analytical_grid)) / np.power(10, amp_analytical_grid)) * 100
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid
    diff_phase_percent = (diff_phase / phase_analytical_grid) * 100

    diff_amp_lim = np.nanmax(np.abs(diff_amp))
    diff_phase_lim = np.nanmax(np.abs(diff_phase))
    diff_amp_percent_lim = 100
    diff_phase_percent_lim = 100

    extent = (xi.min(), xi.max(), yi.min(), yi.max())
    cmap = "RdBu"

    im0 = axs[0, i].imshow(diff_amp, extent=extents, origin='lower', cmap=cmap, vmin=-diff_amp_lim, vmax=diff_amp_lim)
    fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    im1 = axs[1, i].imshow(diff_amp_percent, extent=extents, origin='lower', cmap=cmap, vmin=-diff_amp_percent_lim, vmax=diff_amp_percent_lim)
    fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    im2 = axs[2, i].imshow(diff_phase, extent=extents, origin='lower', cmap=cmap, vmin=-diff_phase_lim, vmax=diff_phase_lim)
    fig.colorbar(im2, ax=axs[2, i], fraction=0.046, pad=0.02)
    im3 = axs[3, i].imshow(diff_phase_percent, extent=extents, origin='lower', cmap=cmap, vmin=-diff_phase_percent_lim, vmax=diff_phase_percent_lim)
    fig.colorbar(im3, ax=axs[3, i], fraction=0.046, pad=0.02)

    for row in range(4):
        axs[row, i].set_xlabel("x", fontsize=16)
        axs[row, i].set_ylabel("y", fontsize=16)
        axs[row, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.suptitle('Difference Between elfe3D and Analytical Solution for Various Air Spaces Above and Below the Source - with PML 1/4th of a Wavelength thick\n2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')

filename_combined = "28_air_space_comparison_diff_75.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)
if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)
fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


## Test for Mimimum Air Space before PML Can be Used - 1.5m

In [4]:
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_ani_def.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

vtk_file_2 = os.path.join(base_folder, "GPR_model_air_10m_025_150.1.vtk")
elfe_vtk_2 = pv.read(vtk_file_2)

vtk_file_3 = os.path.join(base_folder, "GPR_model_air_10m_050_150.1.vtk")
elfe_vtk_3 = pv.read(vtk_file_3)

vtk_file_4 = os.path.join(base_folder, "GPR_model_air_10m_100_150.1.vtk")
elfe_vtk_4 = pv.read(vtk_file_4)

vtk_file_5 = os.path.join(base_folder, "GPR_model_air_10m_150_150.1.vtk")
elfe_vtk_5 = pv.read(vtk_file_5)

vtk_file_6 = os.path.join(base_folder, "GPR_model_air_10m_200_150.1.vtk")
elfe_vtk_6 = pv.read(vtk_file_6)

elfe_vtk_all = [elfe_vtk_1, elfe_vtk_2, elfe_vtk_3, elfe_vtk_4, elfe_vtk_5, elfe_vtk_6]

In [6]:
# Extracting Plot Data
xlim = [-6.5, 6.5]
ylim = [-6.5, 6.5]
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin([np.nanmin(grid) for grid in amp_elfe_grid_all if np.any(~np.isnan(grid))]))
amp_max = np.ceil(np.nanmax([np.nanmax(grid) for grid in amp_elfe_grid_all if np.any(~np.isnan(grid))]))
amp_clim = [amp_min, amp_max]

In [7]:
# Analytical
xi = np.linspace(xlim[0], xlim[1], grid_resolution)
yi = np.linspace(ylim[0], ylim[1], grid_resolution)
grid_x, grid_y = np.meshgrid(xi, yi)

analytical_file = os.path.join(analytical_folder, "air_z0.01_100MHz_xmax5.csv")
analytical_slices = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

x = analytical_slices[:, 0]
y = analytical_slices[:, 1]

x_unique = np.unique(x)
y_unique = np.unique(y)
nx, ny = len(x_unique), len(y_unique)

if len(x) != nx * ny:
    raise ValueError("Mismatch in analytical grid. Expected meshgrid-style structure.")

if component == 'Ex':
    real_index = 4
    imag_index = 5
elif component == 'Ey':
    real_index = 8
    imag_index = 9
elif component == 'Ez':
    real_index = 12
    imag_index = 13
else:
    raise ValueError("Invalid component. Choose from 'Ex', 'Ey', or 'Ez'.")

real_analytical = analytical_slices[:, real_index].reshape((ny, nx))
imag_analytical = analytical_slices[:, imag_index].reshape((ny, nx))

# ----------------------------------------------------------------------
# Extra Process Based on Inference
# ----------------------------------------------------------------------
real_analytical = -1 * real_analytical
imag_analytical = -1 * imag_analytical

complex_analytical = real_analytical + 1j * imag_analytical

# Interpolation
complex_analytical_grid = griddata((x, y), complex_analytical.flatten(), (grid_x, grid_y), method='linear')
amp_analytical_grid = np.log10(np.clip(np.abs(complex_analytical_grid), 1e-60, None))
phase_analytical_grid = np.angle(complex_analytical_grid)  # returns phase in radians (−π to π)
real_analytical_grid = griddata((x, y), real_analytical.flatten(), (grid_x, grid_y), method='linear')
imag_analytical_grid = griddata((x, y), imag_analytical.flatten(), (grid_x, grid_y), method='linear')

#### Plot 29

In [8]:
# --- Plotting ---
extents = (-6.5, 6.5, -6.5, 6.5)  # Set the extents for the images
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'5.0 m, Default', '0.25 m Air', '0.5 m Air', '1.0 m Air', '1.5 m Air', '2.0 m Air']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=extents, origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5,-5), 10, 10, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=extents, origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=2, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=extents, origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=extents, origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Comparison of Various Air Space Thicknesses Above and Below the Source - with PML 1/2th of a Wavelength thick, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"29_air_space_comparison_150.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

#### Plot 28

In [9]:
# --- Plotting ---
clip=4
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'5.0 m, Default', '0.25 m Air', '0.5 m Air', '1.0 m Air', '1.5 m Air', '2.0 m Air']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=extents, origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5,-5),10,10, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=extents, origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=2, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=extents, origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=extents, origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Clipped Comparison of Various Air Space Thicknesses Above and Below the Source - with PML 1/2th of a Wavelength thick, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"29_air_space_comparison_clipped_150.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

#### Plot 30

In [10]:
n_entries = len(elfe_vtk_all)
fig, axs = plt.subplots(4, n_entries, figsize=(6.5 * n_entries, 24), dpi=dpi)

amp_text_list = [r'5.0 m, Default', '0.25 m Air', '0.5 m Air', '1.0 m Air', '1.5 m Air', '2.0 m Air']
for i, label in enumerate(amp_text_list):
    axs[0, i].set_title(f'Log of Ratio of Amplitudes: {label}', fontsize=18, fontweight='bold')
    axs[1, i].set_title(f'Normalised Diff, Amplitudes: {label}', fontsize=18, fontweight='bold')
    axs[2, i].set_title(f'Diff Phase: {label}', fontsize=18, fontweight='bold')
    axs[3, i].set_title(f'% Diff Phase: {label}', fontsize=18, fontweight='bold')

for i in range(n_entries):
    diff_amp = amp_elfe_grid_all[i] - amp_analytical_grid
    diff_amp_percent = ((np.power(10, amp_elfe_grid_all[i]) - np.power(10, amp_analytical_grid)) / np.power(10, amp_analytical_grid)) * 100
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid
    diff_phase_percent = (diff_phase / phase_analytical_grid) * 100

    diff_amp_lim = np.nanmax(np.abs(diff_amp))
    diff_phase_lim = np.nanmax(np.abs(diff_phase))
    diff_amp_percent_lim = 100
    diff_phase_percent_lim = 100

    extent = (xi.min(), xi.max(), yi.min(), yi.max())
    cmap = "RdBu"

    im0 = axs[0, i].imshow(diff_amp, extent=extents, origin='lower', cmap=cmap, vmin=-diff_amp_lim, vmax=diff_amp_lim)
    fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    im1 = axs[1, i].imshow(diff_amp_percent, extent=extents, origin='lower', cmap=cmap, vmin=-diff_amp_percent_lim, vmax=diff_amp_percent_lim)
    fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    im2 = axs[2, i].imshow(diff_phase, extent=extents, origin='lower', cmap=cmap, vmin=-diff_phase_lim, vmax=diff_phase_lim)
    fig.colorbar(im2, ax=axs[2, i], fraction=0.046, pad=0.02)
    im3 = axs[3, i].imshow(diff_phase_percent, extent=extents, origin='lower', cmap=cmap, vmin=-diff_phase_percent_lim, vmax=diff_phase_percent_lim)
    fig.colorbar(im3, ax=axs[3, i], fraction=0.046, pad=0.02)

    for row in range(4):
        axs[row, i].set_xlabel("x", fontsize=16)
        axs[row, i].set_ylabel("y", fontsize=16)
        axs[row, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.suptitle('Difference Between elfe3D and Analytical Solution for Various Air Spaces Above and Below the Source - with PML 1/2th of a Wavelength thick\n2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')

filename_combined = "30_air_space_comparison_diff_150.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)
if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)
fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


#### Making Histogram of distribution of Errors

In [11]:
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_ani_def.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

vtk_file_2 = os.path.join(base_folder, "GPR_model_air_10m_100_150.1.vtk")
elfe_vtk_2 = pv.read(vtk_file_2)

vtk_file_3 = os.path.join(base_folder, "GPR_model_air_10m_150_150.1.vtk")
elfe_vtk_3 = pv.read(vtk_file_3)

vtk_file_4 = os.path.join(base_folder, "GPR_model_air_10m_200_150.1.vtk")
elfe_vtk_4 = pv.read(vtk_file_4)

vtk_file_5 = os.path.join(base_folder, "GPR_model_air_10m_150_75.1.vtk")
elfe_vtk_5 = pv.read(vtk_file_5)

vtk_file_6 = os.path.join(base_folder, "GPR_model_air_10m_200_75.1.vtk")
elfe_vtk_6 = pv.read(vtk_file_6)

elfe_vtk_all = [elfe_vtk_1, elfe_vtk_2, elfe_vtk_3, elfe_vtk_4, elfe_vtk_5, elfe_vtk_6]

In [12]:
# Extracting Plot Data
xlim = [-5, 5]
ylim = [-5, 5]
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin([np.nanmin(grid) for grid in amp_elfe_grid_all if np.any(~np.isnan(grid))]))
amp_max = np.ceil(np.nanmax([np.nanmax(grid) for grid in amp_elfe_grid_all if np.any(~np.isnan(grid))]))
amp_clim = [amp_min, amp_max]

In [13]:
# Analytical
xi = np.linspace(xlim[0], xlim[1], grid_resolution)
yi = np.linspace(ylim[0], ylim[1], grid_resolution)
grid_x, grid_y = np.meshgrid(xi, yi)

analytical_file = os.path.join(analytical_folder, "air_z0.01_100MHz_xmax5.csv")
analytical_slices = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

x = analytical_slices[:, 0]
y = analytical_slices[:, 1]

x_unique = np.unique(x)
y_unique = np.unique(y)
nx, ny = len(x_unique), len(y_unique)

if len(x) != nx * ny:
    raise ValueError("Mismatch in analytical grid. Expected meshgrid-style structure.")

if component == 'Ex':
    real_index = 4
    imag_index = 5
elif component == 'Ey':
    real_index = 8
    imag_index = 9
elif component == 'Ez':
    real_index = 12
    imag_index = 13
else:
    raise ValueError("Invalid component. Choose from 'Ex', 'Ey', or 'Ez'.")

real_analytical = analytical_slices[:, real_index].reshape((ny, nx))
imag_analytical = analytical_slices[:, imag_index].reshape((ny, nx))

# ----------------------------------------------------------------------
# Extra Process Based on Inference
# ----------------------------------------------------------------------
real_analytical = -1 * real_analytical
imag_analytical = -1 * imag_analytical

complex_analytical = real_analytical + 1j * imag_analytical

# Interpolation
complex_analytical_grid = griddata((x, y), complex_analytical.flatten(), (grid_x, grid_y), method='linear')
amp_analytical_grid = np.log10(np.clip(np.abs(complex_analytical_grid), 1e-60, None))
phase_analytical_grid = np.angle(complex_analytical_grid)  # returns phase in radians (−π to π)
real_analytical_grid = griddata((x, y), real_analytical.flatten(), (grid_x, grid_y), method='linear')
imag_analytical_grid = griddata((x, y), imag_analytical.flatten(), (grid_x, grid_y), method='linear')

In [23]:
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
diff_amp_percent_all = []
diff_phase_all = []
std_amp_all = []
std_phase_all = []
amp_text_list = [r'1.5 m PML, 5m Air', '1.5 m PML, 1 m Air', '1.5 m PML, 1.5 m Air', '1.5 m PML, 2 m Air', '0.75 m PML, 1.5 m Air', '0.75 m PML, 2 m Air']
phase_text_list = amp_text_list.copy()
for i in range(n_entries):
    diff_amp_percent = ((np.power(10, amp_elfe_grid_all[i]) - np.power(10, amp_analytical_grid)) / np.power(10, amp_analytical_grid)) * 100
    diff_amp_percent = np.clip(diff_amp_percent, -100, 100)
    diff_phase_percent = (phase_elfe_grid_all[i] - phase_analytical_grid) * 100 
    diff_phase_percent = np.clip(diff_phase_percent, -100, 100)
    std_amp_all.append(np.nanstd(diff_amp_percent))
    std_phase_all.append(np.nanstd(diff_phase_percent))

    diff_amp_percent_all.append(diff_amp_percent)
    diff_phase_all.append(diff_phase_percent)

# Plotting the histograms
fig, axs = plt.subplots(2, n_entries, figsize=(6.5 * n_entries, 14), dpi=dpi)
for i in range(n_entries):
    axs[0, i].hist(diff_amp_percent_all[i].flatten(), bins=100, color='blue', alpha=0.7)
    axs[0, i].set_title(f'Diff Amp Percent: {amp_text_list[i]}', fontsize=18, fontweight='bold')
    axs[0, i].set_xlabel('Difference (%)', fontsize=16)
    axs[0, i].set_ylabel('Frequency', fontsize=16)
    axs[0, i].tick_params(axis='both', which='major', labelsize=16)
    axs[0, i].axvline(std_amp_all[i], color='blue', linestyle='dashed', linewidth=1)
    axs[0, i].axvline(-std_amp_all[i], color='blue', linestyle='dashed', linewidth=1)
    axs[0, i].text(std_amp_all[i], axs[0, i].get_ylim()[1]*0.9, f'STD: {std_amp_all[i]:.2f}', color='blue', fontsize=16)

    axs[1, i].hist(diff_phase_all[i].flatten(), bins=100, color='red', alpha=0.7)
    axs[1, i].set_title(f'Diff Phase: {phase_text_list[i]}', fontsize=18, fontweight='bold')
    axs[1, i].set_xlabel('Difference (%)', fontsize=16)
    axs[1, i].set_ylabel('Frequency', fontsize=16)
    axs[1, i].tick_params(axis='both', which='major', labelsize=16)
    axs[1, i].axvline(std_phase_all[i], color='red', linestyle='dashed', linewidth=1)
    axs[1, i].axvline(-std_phase_all[i], color='red', linestyle='dashed', linewidth=1)
    axs[1, i].text(std_phase_all[i], axs[1, i].get_ylim()[1]*0.9, f'STD: {std_phase_all[i]:.2f}', color='red', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.89])
plt.suptitle('Histogram of Differences Between elfe3D and Analytical Solution for Various Air Spaces Above and Below the Source\n2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')
plt.savefig(os.path.join(postprocess_folder, "31_air_space_comparison_histogram.png"), dpi=dpi)
plt.close(fig)


In [ ]:
n_entries = len(elfe_vtk_all)
fig, axs = plt.subplots(4, n_entries, figsize=(6.5 * n_entries, 24), dpi=dpi)

amp_text_list = [r'5.0 m, Default', '0.25 m Air', '0.5 m Air', '1.0 m Air', '1.5 m Air', '2.0 m Air']
for i, label in enumerate(amp_text_list):
    axs[0, i].set_title(f'Log of Ratio of Amplitudes: {label}', fontsize=18, fontweight='bold')
    axs[1, i].set_title(f'Normalised Diff, Amplitudes: {label}', fontsize=18, fontweight='bold')
    axs[2, i].set_title(f'Diff Phase: {label}', fontsize=18, fontweight='bold')
    axs[3, i].set_title(f'% Diff Phase: {label}', fontsize=18, fontweight='bold')

for i in range(n_entries):
    diff_amp = amp_elfe_grid_all[i] - amp_analytical_grid
    diff_amp_percent = ((np.power(10, amp_elfe_grid_all[i]) - np.power(10, amp_analytical_grid)) / np.power(10, amp_analytical_grid)) * 100
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid
    diff_phase_percent = (diff_phase / phase_analytical_grid) * 100

    diff_amp_lim = np.nanmax(np.abs(diff_amp))
    diff_phase_lim = np.nanmax(np.abs(diff_phase))
    diff_amp_percent_lim = 100
    diff_phase_percent_lim = 100

    extent = (xi.min(), xi.max(), yi.min(), yi.max())
    cmap = "RdBu"

    im0 = axs[0, i].imshow(diff_amp, extent=extents, origin='lower', cmap=cmap, vmin=-diff_amp_lim, vmax=diff_amp_lim)
    fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    im1 = axs[1, i].imshow(diff_amp_percent, extent=extents, origin='lower', cmap=cmap, vmin=-diff_amp_percent_lim, vmax=diff_amp_percent_lim)
    fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    im2 = axs[2, i].imshow(diff_phase, extent=extents, origin='lower', cmap=cmap, vmin=-diff_phase_lim, vmax=diff_phase_lim)
    fig.colorbar(im2, ax=axs[2, i], fraction=0.046, pad=0.02)
    im3 = axs[3, i].imshow(diff_phase_percent, extent=extents, origin='lower', cmap=cmap, vmin=-diff_phase_percent_lim, vmax=diff_phase_percent_lim)
    fig.colorbar(im3, ax=axs[3, i], fraction=0.046, pad=0.02)

    for row in range(4):
        axs[row, i].set_xlabel("x", fontsize=16)
        axs[row, i].set_ylabel("y", fontsize=16)
        axs[row, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.suptitle('Difference Between elfe3D and Analytical Solution for Various Air Spaces Above and Below the Source - with PML 1/2th of a Wavelength thick\n2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')

filename_combined = "30_air_space_comparison_diff_150.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)
if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)
fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


## Test for Layered Model - Halfspace: Air 1m, eps_r 4 3m, PML 0.75m / half lambda - 0.005 m3 volume

In [15]:
electric_file = os.path.join(base_folder, "out_4_kmax", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_kmax_10m", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_antenna025_mindomain", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_antenna025_150", "electric_fields_receiver_line.txt")
e_data_elfe_4 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3, e_data_elfe_4]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

In [16]:
# Analytical Data
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)


#### Plots 35, 36, 37

In [21]:
def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    for abs_data, phase_data, rec_xi, rec_yi, label in zip(abs_data_list, phase_data_list, rec_x, rec_y, labels):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data, linestyle='-', marker='o', label=label)
        # Phase plot
        axes[1].plot(rec_distance, phase_data, linestyle='-', marker='o', label=label)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0].plot(rec_distance_an, abs_an, linestyle='--', color='black', label='Analytical Solution')
    axes[1].plot(rec_distance_an, phase_an, linestyle='--', color='black', label='Analytical Solution')

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)
    # axes[1].legend(loc='upper right')

    plt.subplots_adjust(top=0.82)  # Give more space for the suptitle
    title = r'Half-Space Model Response - $\epsilon_r = 4$ & $\sigma = 1e$-4,' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

labels = [r'Domain: 6m Centred, on Ground', r'Domain: 10m Centred, On Ground', r'Domain: 6m, One Quadrant, 2.5 cm Above', 'Domain: 6.5m, One Quadrant, 2.5 cm Above']

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs[:-1])
    rec_y_bs_list.append(rec_y_bs[:-1])
    abs_Ex_bs_list.append(abs_Ex_bs[:-1])
    phase_Ex_bs_list.append(phase_Ex_bs[:-1])

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef[:-1])
    rec_y_ef_list.append(rec_y_ef[:-1])
    abs_Ex_ef_list.append(abs_Ex_ef[:-1])
    phase_Ex_ef_list.append(phase_Ex_ef[:-1])

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob[:-1])
    rec_y_ob_list.append(rec_y_ob[:-1])
    abs_Ex_ob_list.append(abs_Ex_ob[:-1])
    phase_Ex_ob_list.append(phase_Ex_ob[:-1])

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r
y_an_ef = np.zeros_like(r)
abs_Ex_an_ef = abs_broadside
phase_Ex_an_ef = phase_broadside

x_an_bs = np.zeros_like(r)
y_an_bs = r
abs_Ex_an_bs = abs_endfire
phase_Ex_an_bs = phase_endfire

x_an_ob = r
y_an_ob = r
abs_Ex_an_ob = abs_45deg
phase_Ex_an_ob = phase_45deg

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 35, '4', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 36, '4', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 37, '4', labels)


## Test for Layered Model - 2 Layers: Air 1m, eps_r 4 2m, 9 1 m, PML 0.5m / half lambda - 0.002 m3 volume

In [30]:
electric_file = os.path.join(base_folder, "out_4_9_kmax_10m", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_antenna025_mindomain", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_antenna025_150", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)


labels = [r'Domain: 10m, On Ground', 
          'One Quadrant, 2.5 cm Above', 
          'One Quadrant, 2.5 cm, 1.5m PML', 
          'One Quadrant, 0.0001 m3',
          'One Quadrant, 0.75m PML, 0.0002 m3',
          ]


e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_9_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    for abs_data, phase_data, rec_xi, rec_yi, label in zip(abs_data_list, phase_data_list, rec_x, rec_y, labels):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data, linestyle='-', marker='o', label=label)
        # Phase plot
        axes[1].plot(rec_distance, phase_data, linestyle='-', marker='o', label=label)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0].plot(rec_distance_an, abs_an, linestyle='--', color='black', label='Analytical Solution')
    axes[1].plot(rec_distance_an, phase_an, linestyle='--', color='black', label='Analytical Solution')

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)
    # axes[1].legend(loc='upper right')

    plt.subplots_adjust(top=0.82)  # Give more space for the suptitle
    title = r'Two-Layered Model Response - 2 m Layer of $\epsilon_r = 4$ with Half-space of $\epsilon_r = 9$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs[:-1])
    rec_y_bs_list.append(rec_y_bs[:-1])
    abs_Ex_bs_list.append(abs_Ex_bs[:-1])
    phase_Ex_bs_list.append(phase_Ex_bs[:-1])

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef[:-1])
    rec_y_ef_list.append(rec_y_ef[:-1])
    abs_Ex_ef_list.append(abs_Ex_ef[:-1])
    phase_Ex_ef_list.append(phase_Ex_ef[:-1])

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob[:-1])
    rec_y_ob_list.append(rec_y_ob[:-1])
    abs_Ex_ob_list.append(abs_Ex_ob[:-1])
    phase_Ex_ob_list.append(phase_Ex_ob[:-1])

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r
y_an_ef = np.zeros_like(r)
abs_Ex_an_ef = abs_broadside
phase_Ex_an_ef = phase_broadside

x_an_bs = np.zeros_like(r)
y_an_bs = r
abs_Ex_an_bs = abs_endfire
phase_Ex_an_bs = phase_endfire

x_an_ob = r
y_an_ob = r
abs_Ex_an_ob = abs_45deg
phase_Ex_an_ob = phase_45deg

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 42, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 43, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 44, '4_9', labels)

In [33]:
electric_file = os.path.join(base_folder, "out_4_9_kmax_10m", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_finer", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_thicker", "electric_fields_receiver_line.txt")
e_data_elfe_4 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_3, e_data_elfe_4]

labels = [r'Domain: 10m, On Ground, 0.5m PML, 0.0002 m3', 
          'One Quadrant, 2.5 cm Above, 0.5m PML, 0.0001 m3',
          'One Quadrant, 2.5 cm Above, 0.75m PML, 0.0002 m3',
          ]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_9_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    for abs_data, phase_data, rec_xi, rec_yi, label in zip(abs_data_list, phase_data_list, rec_x, rec_y, labels):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data, linestyle='-', marker='o', label=label)
        # Phase plot
        axes[1].plot(rec_distance, phase_data, linestyle='-', marker='o', label=label)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0].plot(rec_distance_an, abs_an, linestyle='--', color='black', label='Analytical Solution')
    axes[1].plot(rec_distance_an, phase_an, linestyle='--', color='black', label='Analytical Solution')

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)
    # axes[1].legend(loc='upper right')

    plt.subplots_adjust(top=0.82)  # Give more space for the suptitle
    title = r'Two-Layered Model Response - 2 m Layer of $\epsilon_r = 4$ with Half-space of $\epsilon_r = 9$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs[:-1])
    rec_y_bs_list.append(rec_y_bs[:-1])
    abs_Ex_bs_list.append(abs_Ex_bs[:-1])
    phase_Ex_bs_list.append(phase_Ex_bs[:-1])

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef[:-1])
    rec_y_ef_list.append(rec_y_ef[:-1])
    abs_Ex_ef_list.append(abs_Ex_ef[:-1])
    phase_Ex_ef_list.append(phase_Ex_ef[:-1])

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob[:-1])
    rec_y_ob_list.append(rec_y_ob[:-1])
    abs_Ex_ob_list.append(abs_Ex_ob[:-1])
    phase_Ex_ob_list.append(phase_Ex_ob[:-1])

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r
y_an_ef = np.zeros_like(r)
abs_Ex_an_ef = abs_broadside
phase_Ex_an_ef = phase_broadside

x_an_bs = np.zeros_like(r)
y_an_bs = r
abs_Ex_an_bs = abs_endfire
phase_Ex_an_bs = phase_endfire

x_an_ob = r
y_an_ob = r
abs_Ex_an_ob = abs_45deg
phase_Ex_an_ob = phase_45deg

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 46, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 47, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 48, '4_9', labels)

## Second Order Test 1

In [2]:
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_air_sok.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

elfe_vtk_all = [elfe_vtk_1]

# Extracting Plot Data
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin(amp_elfe_grid_all[0]))
amp_max = np.ceil(np.nanmax(amp_elfe_grid_all[0]))
amp_clim = [amp_min, amp_max]

# Analytical
xi = np.linspace(xlim[0], xlim[1], grid_resolution)
yi = np.linspace(ylim[0], ylim[1], grid_resolution)
grid_x, grid_y = np.meshgrid(xi, yi)

analytical_file = os.path.join(analytical_folder, "air_z0.01_100MHz_xmax5.csv")
analytical_slices = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

x = analytical_slices[:, 0]
y = analytical_slices[:, 1]

x_unique = np.unique(x)
y_unique = np.unique(y)
nx, ny = len(x_unique), len(y_unique)

if len(x) != nx * ny:
    raise ValueError("Mismatch in analytical grid. Expected meshgrid-style structure.")

if component == 'Ex':
    real_index = 4
    imag_index = 5
elif component == 'Ey':
    real_index = 8
    imag_index = 9
elif component == 'Ez':
    real_index = 12
    imag_index = 13
else:
    raise ValueError("Invalid component. Choose from 'Ex', 'Ey', or 'Ez'.")

real_analytical = analytical_slices[:, real_index].reshape((ny, nx))
imag_analytical = analytical_slices[:, imag_index].reshape((ny, nx))

# ----------------------------------------------------------------------
# Extra Process Based on Inference
# ----------------------------------------------------------------------
real_analytical = -1 * real_analytical
imag_analytical = -1 * imag_analytical

complex_analytical = real_analytical + 1j * imag_analytical

# Interpolation
complex_analytical_grid = griddata((x, y), complex_analytical.flatten(), (grid_x, grid_y), method='linear')
amp_analytical_grid = np.log10(np.clip(np.abs(complex_analytical_grid), 1e-60, None))
phase_analytical_grid = np.angle(complex_analytical_grid)  # returns phase in radians (−π to π)
real_analytical_grid = griddata((x, y), real_analytical.flatten(), (grid_x, grid_y), method='linear')
imag_analytical_grid = griddata((x, y), imag_analytical.flatten(), (grid_x, grid_y), method='linear')

# --- Plotting ---
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = ['Second Order, Air']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Second Order PML With the Simple Exact Decay Function, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"41_sok_air.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

## Test for Var

In [2]:
xlim = [-3, 6.5]
ylim = [-3, 6.5]
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_ani_def.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

vtk_file_2 = os.path.join(base_folder, "GPR_model_air_ref_berenger.1.vtk")
elfe_vtk_2 = pv.read(vtk_file_2)

vtk_file_3 = os.path.join(base_folder, "GPR_model_air_ref_berenger2.1.vtk")
elfe_vtk_3 = pv.read(vtk_file_3)

vtk_file_4 = os.path.join(base_folder, "GPR_model_air_ref_feng.1.vtk")
elfe_vtk_4 = pv.read(vtk_file_4)

vtk_file_5 = os.path.join(base_folder, "GPR_model_air_ref_irv.1.vtk")
elfe_vtk_5 = pv.read(vtk_file_5)

elfe_vtk_all = [elfe_vtk_1, elfe_vtk_2, elfe_vtk_3, elfe_vtk_4, elfe_vtk_5]

# Extracting Plot Data
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin(amp_elfe_grid_all[0]))
amp_max = np.ceil(np.nanmax(amp_elfe_grid_all[0]))
amp_clim = [amp_min, amp_max]

# Analytical
xi = np.linspace(xlim[0], xlim[1], grid_resolution)
yi = np.linspace(ylim[0], ylim[1], grid_resolution)
grid_x, grid_y = np.meshgrid(xi, yi)

analytical_file = os.path.join(analytical_folder, "air_z0.01_100MHz_xmax5.csv")
analytical_slices = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

x = analytical_slices[:, 0]
y = analytical_slices[:, 1]

x_unique = np.unique(x)
y_unique = np.unique(y)
nx, ny = len(x_unique), len(y_unique)

if len(x) != nx * ny:
    raise ValueError("Mismatch in analytical grid. Expected meshgrid-style structure.")

if component == 'Ex':
    real_index = 4
    imag_index = 5
elif component == 'Ey':
    real_index = 8
    imag_index = 9
elif component == 'Ez':
    real_index = 12
    imag_index = 13
else:
    raise ValueError("Invalid component. Choose from 'Ex', 'Ey', or 'Ez'.")

real_analytical = analytical_slices[:, real_index].reshape((ny, nx))
imag_analytical = analytical_slices[:, imag_index].reshape((ny, nx))

# ----------------------------------------------------------------------
# Extra Process Based on Inference
# ----------------------------------------------------------------------
real_analytical = -1 * real_analytical
imag_analytical = -1 * imag_analytical

complex_analytical = real_analytical + 1j * imag_analytical

# Interpolation
complex_analytical_grid = griddata((x, y), complex_analytical.flatten(), (grid_x, grid_y), method='linear')
amp_analytical_grid = np.log10(np.clip(np.abs(complex_analytical_grid), 1e-60, None))
phase_analytical_grid = np.angle(complex_analytical_grid)  # returns phase in radians (−π to π)
real_analytical_grid = griddata((x, y), real_analytical.flatten(), (grid_x, grid_y), method='linear')
imag_analytical_grid = griddata((x, y), imag_analytical.flatten(), (grid_x, grid_y), method='linear')

# --- Plotting ---
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = ['Anisotropic, Default', 'Berenger - kmax+eps_max', 'Berenger - kmax+eps_var', 'Feng', 'Irving']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-3, -3), 9.5, 9.5, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-3, -3), 9.5, 9.5, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Comparison of Various PML Implementation Strategies With the Simple Exact Decay Function, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"49_theory_numerical_comparison.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

In [5]:
n_entries = len(elfe_vtk_all)
fig, axs = plt.subplots(4, n_entries, figsize=(6.5 * n_entries, 24), dpi=dpi)

amp_text_list = ['Anisotropic, Default', 'Berenger - kmax+eps_max', 'Berenger - kmax+eps_var', 'Feng']
for i, label in enumerate(amp_text_list):
    axs[0, i].set_title(f'Diff Log Amplitude: {label}', fontsize=18, fontweight='bold')
    axs[1, i].set_title(f'% Diff Log Amp: {label}', fontsize=18, fontweight='bold')
    axs[2, i].set_title(f'Diff Phase: {label}', fontsize=18, fontweight='bold')
    axs[3, i].set_title(f'% Diff Phase: {label}', fontsize=18, fontweight='bold')

for i in range(n_entries):
    diff_amp = amp_elfe_grid_all[i] - amp_analytical_grid
    diff_amp_percent = (diff_amp / amp_analytical_grid) * 100
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid
    diff_phase_percent = (diff_phase / phase_analytical_grid) * 100

    diff_amp_lim = np.nanmax(np.abs(diff_amp))
    diff_phase_lim = np.nanmax(np.abs(diff_phase))
    diff_amp_percent_lim = 100
    diff_phase_percent_lim = 100

    extent = (xi.min(), xi.max(), yi.min(), yi.max())
    cmap = "RdBu"

    im0 = axs[0, i].imshow(diff_amp, extent=extent, origin='lower', cmap=cmap, vmin=-diff_amp_lim, vmax=diff_amp_lim)
    fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    im1 = axs[1, i].imshow(diff_amp_percent, extent=extent, origin='lower', cmap=cmap, vmin=-diff_amp_percent_lim, vmax=diff_amp_percent_lim)
    fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    im2 = axs[2, i].imshow(diff_phase, extent=extent, origin='lower', cmap=cmap, vmin=-diff_phase_lim, vmax=diff_phase_lim)
    fig.colorbar(im2, ax=axs[2, i], fraction=0.046, pad=0.02)
    im3 = axs[3, i].imshow(diff_phase_percent, extent=extent, origin='lower', cmap=cmap, vmin=-diff_phase_percent_lim, vmax=diff_phase_percent_lim)
    fig.colorbar(im3, ax=axs[3, i], fraction=0.046, pad=0.02)

    for row in range(4):
        axs[row, i].set_xlabel("x", fontsize=16)
        axs[row, i].set_ylabel("y", fontsize=16)
        axs[row, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.suptitle('Difference Between elfe3D and Analytical Solution for All Implementation Strategies\n2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')

filename_combined = "50_all_methods_analytic_diff.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)
if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)
fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


In [6]:
# --- Plotting ---
clip=4
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = ['Anisotropic, Default', 'Berenger - kmax+eps_max', 'Berenger - kmax+eps_var', 'Feng']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xi.min(), xi.max(), yi.min(), yi.max()), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-5, -5), 10, 10, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(-5, 5, -5, 5), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Clipped Comparison of Various PML Implementation Strategies With the Simple Exact Decay Function, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"51_theory_numerical_comparison_clipped.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

Feng and Berenger

In [6]:
electric_file = os.path.join(base_folder, "out_4_ref_Berenger", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_ref_Berenger2", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_ref_Feng", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_kmax_10m", "electric_fields_receiver_line.txt")
e_data_elfe_0 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_0, e_data_elfe_1, e_data_elfe_2, e_data_elfe_3]

labels = ['kmax+eps_max, Default 10m',
          'Berenger - kmax+eps_max',
          'Berenger - kmax+eps_var',
          'Feng',
          ]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    for abs_data, phase_data, rec_xi, rec_yi, label in zip(abs_data_list, phase_data_list, rec_x, rec_y, labels):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data, linestyle='--', marker='o', label=label)
        # Phase plot
        axes[1].plot(rec_distance, phase_data, linestyle='--', marker='o', label=label)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0].plot(rec_distance_an, abs_an, linestyle='-', color='black', label='Analytical Solution')
    axes[1].plot(rec_distance_an, phase_an, linestyle='-', color='black', label='Analytical Solution')

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)
    # axes[1].legend(loc='upper right')

    plt.subplots_adjust(top=0.82)  # Give more space for the suptitle
    title = r'Half-Space Model Response - Half-space of $\epsilon_r = 4$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs[:-1])
    rec_y_bs_list.append(rec_y_bs[:-1])
    abs_Ex_bs_list.append(abs_Ex_bs[:-1])
    phase_Ex_bs_list.append(phase_Ex_bs[:-1])

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef[:-1])
    rec_y_ef_list.append(rec_y_ef[:-1])
    abs_Ex_ef_list.append(abs_Ex_ef[:-1])
    phase_Ex_ef_list.append(phase_Ex_ef[:-1])

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob[:-1])
    rec_y_ob_list.append(rec_y_ob[:-1])
    abs_Ex_ob_list.append(abs_Ex_ob[:-1])
    phase_Ex_ob_list.append(phase_Ex_ob[:-1])

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r
y_an_ef = np.zeros_like(r)
abs_Ex_an_ef = abs_broadside
phase_Ex_an_ef = phase_broadside

x_an_bs = np.zeros_like(r)
y_an_bs = r
abs_Ex_an_bs = abs_endfire
phase_Ex_an_bs = phase_endfire

x_an_ob = r
y_an_ob = r
abs_Ex_an_ob = abs_45deg
phase_Ex_an_ob = phase_45deg

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 52, '4', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 53, '4', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 54, '4', labels)

In [5]:
electric_file = os.path.join(base_folder, "out_4_9_ref_Berenger", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_ref_Berenger2", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_ref_Feng", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_kmax_10m", "electric_fields_receiver_line.txt")
e_data_elfe_0 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_0, e_data_elfe_1, e_data_elfe_2, e_data_elfe_3]

labels = ['kmax+eps_max, Default 10m',
          'Berenger - kmax+eps_max',
          'Berenger - kmax+eps_var',
          'Feng',
          ]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_9_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    for abs_data, phase_data, rec_xi, rec_yi, label in zip(abs_data_list, phase_data_list, rec_x, rec_y, labels):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data, linestyle='--', marker='o', label=label)
        # Phase plot
        axes[1].plot(rec_distance, phase_data, linestyle='--', marker='o', label=label)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0].plot(rec_distance_an, abs_an, linestyle='-', color='black', label='Analytical Solution')
    axes[1].plot(rec_distance_an, phase_an, linestyle='-', color='black', label='Analytical Solution')

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)
    # axes[1].legend(loc='upper right')

    plt.subplots_adjust(top=0.82)  # Give more space for the suptitle
    title = r'Two-Layered Model Response - 2 m Layer of $\epsilon_r = 4$ with Half-space of $\epsilon_r = 9$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs[:-1])
    rec_y_bs_list.append(rec_y_bs[:-1])
    abs_Ex_bs_list.append(abs_Ex_bs[:-1])
    phase_Ex_bs_list.append(phase_Ex_bs[:-1])

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef[:-1])
    rec_y_ef_list.append(rec_y_ef[:-1])
    abs_Ex_ef_list.append(abs_Ex_ef[:-1])
    phase_Ex_ef_list.append(phase_Ex_ef[:-1])

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob[:-1])
    rec_y_ob_list.append(rec_y_ob[:-1])
    abs_Ex_ob_list.append(abs_Ex_ob[:-1])
    phase_Ex_ob_list.append(phase_Ex_ob[:-1])

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r
y_an_ef = np.zeros_like(r)
abs_Ex_an_ef = abs_broadside
phase_Ex_an_ef = phase_broadside

x_an_bs = np.zeros_like(r)
y_an_bs = r
abs_Ex_an_bs = abs_endfire
phase_Ex_an_bs = phase_endfire

x_an_ob = r
y_an_ob = r
abs_Ex_an_ob = abs_45deg
phase_Ex_an_ob = phase_45deg

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 55, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 56, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 57, '4_9', labels)